# news_track — 리스트 추출 결과로 사건 묶음·후속 보도·매체 수 갱신

`news_extract_tmp.ipynb`가 `source/live/articles.jsonl`에 쌓은 리스트 추출 결과를 읽어, **아직 처리하지 않은 기사만** 회차 순서대로 처리하고 저장된 결과를 갱신한다.

```
news_collect.ipynb  ─ 검색어마다 기사 URL을 DB에 쌓는다(이 노트북은 읽지 않는다)
   ▼
news_extract_tmp.ipynb  ─ 새 기사 찾기 → 원문 받기 → 숫자 영어 변환 → 리스트 추출
   ▼  source/live/articles.jsonl
news_track_tmp.ipynb    ─ 사건 묶음 → 후속 보도 판별 → 매체 수
   ▼  source/live/stories.csv, followup.csv, importance.csv
```

| 절 | 하는 일 | 바탕이 된 코드 |
|---|---|---|
| 3 | 사건 묶음 갱신 | proj `P3_사건묶기_tmp.ipynb`를 회차마다 갱신하도록 바꾼 것 |
| 4 | 후속 보도 판별 | proj `P4_후속보도판별_tmp.ipynb`, 프롬프트 `prompt/followup_prompt.md` |
| 5 | 매체 수 갱신 | proj `P5_매체수_tmp.ipynb` |

- 이 노트북은 `articles.jsonl`을 읽기만 하고, `source/live/`의 `stories.csv`, `story_merges.csv`, `followup.csv`, `followup_state.json`, `importance.csv`를 쓴다.
- 후속 판별에 쓰는 본문은 `news_extract`가 `pending_bodies/<key>.txt`에 잠시 둔 것을 읽고, 쓴 뒤 지운다(4절 `clean_bodies`).
- 사건 묶음·후속 판별은 아직 하지 않은 기사만 하므로, 멈췄다가 다시 켜도 이어서 한다.

## 1. 설정

- `RUN_LLM = False`이면 처리할 기사 수만 보여 주고 **임베딩·LLM을 부르지 않는다.**
- 키는 proj 폴더 `.env`의 `OPENAI_API_KEY`(후속 판별), `GEMINI_API_KEY`(임베딩)이며 값은 출력하지 않는다.

In [ ]:
import datetime as dt
import hashlib
import json
import re
import time
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더(노트북 폴더에서 위로 올라가며 처음 나오는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # news_extract·news_track이 함께 쓰는 결과 폴더
FOLLOWUP_PROMPT_PATH = ROOT / 'prompt' / 'followup_prompt.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 Gemini(임베딩)와 OpenAI(후속 판별)를 부른다
SIM_HIGH = 0.90          # comparison_summary 유사도가 이 이상이면 같은 사건(P3에서 정한 값)
COMPARE_DAYS = 2         # 발행 시각이 이 날수 안에 있는 사건 기사와만 비교
BIG_STORY = 15           # 사건 기사가 이 수 이상이면 합치기가 연쇄로 이어졌는지 보라고 알린다
YAHOO = 'finance.yahoo.com'
WATCH_RUN = False        # True면 마지막 칸에서 WATCH_SECONDS초마다 새 추출 결과를 확인하며 계속 처리
WATCH_SECONDS = 60
# ==============================

ARTICLES_PATH = RESULT_DIR / 'articles.jsonl'        # news_extract가 쌓는 기사(읽기만)
PENDING_BODY_DIR = RESULT_DIR / 'pending_bodies'      # news_extract가 둔 사건 기사 본문(임시, 후속 판별에 쓴 뒤 지운다)
STORIES_PATH = RESULT_DIR / 'stories.csv'            # 기사 → 사건 번호
STORY_MERGES_PATH = RESULT_DIR / 'story_merges.csv'  # 사건을 합칠 때마다 한 줄(언제, 어떤 기사 때문에, 어느 사건을)
FOLLOWUP_PATH = RESULT_DIR / 'followup.csv'          # 기사 → 후속 판단, 소식 번호, 새 사실
FOLLOWUP_STATE_PATH = RESULT_DIR / 'followup_state.json'   # 사건마다 알려진 사실 목록 등(후속 판별의 상태)
IMPORTANCE_PATH = RESULT_DIR / 'importance.csv'      # 소식마다 소식 매체 수·사건 매체 수
EMBED_CACHE_PATH = RESULT_DIR / 'embed_cache_gemini-embedding-001_768.npz'
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'            # LLM 호출마다 토큰 수(news_extract와 함께 씀)
TRACK_LOG_PATH = RESULT_DIR / 'track_log.csv'        # 처리할 때마다 한 줄
LLM_SOURCE_PATH = RESULT_DIR / 'llm_source.jsonl'    # news_extract가 쌓는 기사 요약 — 여기서 story_id·news_outlets를 채운다
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL, EMBED_TASK, EMBED_DIM = 'gemini-embedding-001', 'SEMANTIC_SIMILARITY', 768
assert ROOT.name == 'news_search' and (ROOT / 'news_collect.ipynb').exists(), 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

- `env_value`: proj 폴더 `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `read_jsonl`: `news_extract`가 쓰는 중인 마지막 줄은 건너뛴다(다음 처리 때 읽는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다.
- `embed`: Gemini 임베딩(P3와 같은 설정). 같은 문장은 캐시에서 꺼낸다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def read_csv(path, columns):
    return pd.read_csv(path, dtype=str).fillna('') if path.exists() else pd.DataFrame(columns=columns)


def append_csv(rows, path):
    if rows:
        data = pd.DataFrame(rows)
        if path.exists():   # 이미 있는 파일의 열 순서에 맞춘다(합치기 때 파일을 다시 쓰면 열 순서가 바뀔 수 있다)
            data = data.reindex(columns=pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
        data.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


def same_title(title):
    # 제목 비교용: 소문자로 바꾸고 따옴표·문장부호·공백 차이를 없앤다.
    return re.sub(r'\W+', '', (title or '').lower())


class LLMError(Exception):
    pass


def ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


def log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


EMBED_RETRIES = 30       # 429(분당 한도)면 65초 쉬고 다시 하는 최대 횟수(2026-11-20, 5→30: 밀린 기사를 따라잡을 때 한도에 몰려도 멈추지 않게)


def embed(texts):
    # 캐시에 없는 문장만 요청하고, 입력 순서대로 길이 1인 벡터를 돌려준다(내적 = 코사인 유사도).
    cache_key = lambda t: hashlib.sha1(f'{EMBED_TASK}|{t}'.encode('utf-8')).hexdigest()
    cache = {}
    if EMBED_CACHE_PATH.exists():
        with np.load(EMBED_CACHE_PATH) as z:
            cache = dict(zip(z['keys'].tolist(), z['vecs']))
    todo = sorted({t for t in texts if cache_key(t) not in cache})
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{EMBED_MODEL}:batchEmbedContents'

    def save():
        keys = list(cache)
        np.savez_compressed(EMBED_CACHE_PATH, keys=np.array(keys), vecs=np.stack([cache[k] for k in keys]))

    for i in range(0, len(todo), 50):          # 무료 등급은 분당 100문장이라 50개씩, 429면 65초 쉰다(최대 30번)
        chunk = todo[i:i + 50]
        body = {'requests': [{'model': f'models/{EMBED_MODEL}', 'content': {'parts': [{'text': t}]},
                              'taskType': EMBED_TASK, 'outputDimensionality': EMBED_DIM} for t in chunk]}
        for attempt in range(EMBED_RETRIES):
            request = urllib.request.Request(url, data=json.dumps(body).encode('utf-8'), method='POST',
                                             headers={'x-goog-api-key': env_value('GEMINI_API_KEY') or '', 'Content-Type': 'application/json'})
            try:
                with urllib.request.urlopen(request, timeout=120) as response:
                    vecs = np.array([e['values'] for e in json.loads(response.read())['embeddings']], dtype=np.float32)
                break
            except urllib.error.HTTPError as e:
                if e.code == 429 and attempt < EMBED_RETRIES - 1:
                    print('  임베딩 분당 한도 초과, 65초 뒤 다시')
                    time.sleep(65)
                else:
                    raise LLMError(f'Gemini HTTP {e.code}') from None
        for text, vec in zip(chunk, vecs / np.linalg.norm(vecs, axis=1, keepdims=True)):
            cache[cache_key(text)] = vec
        save()                                  # 50개마다 저장: 중간에 막혀도 받은 것은 남는다(2026-11-20)
    return {t: cache[cache_key(t)] for t in texts}


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정',
      '/ GEMINI_API_KEY:', '설정됨' if env_value('GEMINI_API_KEY') else '미설정')

## 3. 사건 묶음 갱신

`articles.jsonl`에서 추출은 했지만 아직 사건 번호가 없는 기사를 회차·발행 시각 순서로 하나씩 처리한다.

- `news_kind = no_event`(분석·추천 글)는 사건으로 묶지 않는다(`group = 분석·추천 글`).
- 사건 기사는 **발행 시각이 `COMPARE_DAYS`일 안인** 저장된 사건 기사들과 비교해, 붙을 사건 후보를 모은다.
  - **제목 규칙**: 제목이 같은 기사(다른 매체 재게재)가 있으면 그 기사의 사건
  - **유사도**: `comparison_summary` 유사도가 `SIM_HIGH`(0.90) 이상인 기사가 있는 사건
- 후보가 없으면 새 사건 번호를 만든다(`S00001`부터). 후보가 하나면 그 사건에 붙인다.
- **후보가 둘 이상이면 사건을 합친다.** 가장 먼저 생긴 사건 번호로 합치고, 앞 기사들의 사건 번호도 바꾼다(`stories.csv`를 다시 쓴다).
  - 후속 판별 상태도 함께 합친다(4절 `merge_followup`): 뒤 사건의 소식 번호는 이어지는 번호로 바뀐다(예: 앞 사건 소식이 2개면 N1 → N3).
  - 합친 기록은 `story_merges.csv`에 남는다. 사건 기사가 `BIG_STORY`건 이상이 되면 연쇄로 합쳐졌는지 보라고 알린다.
- `stories.csv`의 `joined_by`: `새 사건`, `유사도`, `제목`(제목 규칙으로 붙음), `best_sim`: 붙은 사건 기사들과의 가장 높은 유사도.
- 11-16 자료 시험에서 이 방식은 하루치 전체를 한 번에 묶은 P3와 같은 결과를 냈다(`tmp_results/grouping_compare_20261116/`).


In [ ]:
STORY_COLUMNS = ['key', 'story_id', 'group', 'best_sim', 'joined_by']
MERGE_COLUMNS = ['time_utc', 'key', 'from_story', 'into_story']


def load_articles():
    articles = pd.DataFrame(read_jsonl(ARTICLES_PATH))
    if articles.empty:
        return articles
    # 같은 기사는 마지막 줄만(2026-11-23): news_extract가 원문 받기 실패 기사를 다시 시도하면 줄이 하나 더 쌓인다.
    # 여러 줄을 다 읽으면 기사 번호가 겹치고 예전 실패 줄의 빈 요약이 임베딩에 섞여 Gemini 400으로 사건 묶기가 멈췄다. news_to_db도 마지막 줄을 쓴다
    articles = articles.drop_duplicates('key', keep='last').reset_index(drop=True)
    profile = [(e or {}).get('article_profile') or {} for e in articles['extraction']]
    articles['news_kind'] = [p.get('news_kind') for p in profile]
    articles['short_summary'] = [p.get('short_summary') for p in profile]
    articles['comparison_summary'] = [((e or {}).get('comparison_summary') or '').strip() for e in articles['extraction']]
    articles['published'] = pd.to_datetime(articles['published_utc'], utc=True, format='ISO8601')
    return articles


def update_stories():
    articles = load_articles()
    stories = read_csv(STORIES_PATH, STORY_COLUMNS)
    if articles.empty:
        return set()
    todo = articles[articles['extraction'].notna() & ~articles['key'].isin(stories['key'])].sort_values(['round', 'published'])
    if todo.empty:
        return set()

    # 비교 대상: 이미 사건 번호가 있는 사건 기사 중 이번 기사들과 발행 시각이 COMPARE_DAYS일 안인 것
    window = pd.Timedelta(days=COMPARE_DAYS)
    old = articles.merge(stories[stories['story_id'] != ''], on='key')
    old = old[(old['published'] >= todo['published'].min() - window) & (old['published'] <= todo['published'].max() + window)]
    is_event = (todo['news_kind'] == 'event') & (todo['comparison_summary'] != '')
    vectors = embed(old['comparison_summary'].tolist() + todo.loc[is_event, 'comparison_summary'].tolist())
    pool = [{'story_id': r['story_id'], 'published': r['published'], 'title_key': same_title(r['title']),
             'vec': vectors[r['comparison_summary']]} for _, r in old.sort_values('published').iterrows()]
    story_no = max([int(s[1:]) for s in stories['story_id'] if s] or [0])
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')

    rows, touched, merges = [], set(), []
    for _, r in todo.iterrows():
        if r['news_kind'] != 'event' or not r['comparison_summary']:
            rows.append({'key': r['key'], 'story_id': '', 'group': '분석·추천 글', 'best_sim': '', 'joined_by': ''})
            continue
        vec = vectors[r['comparison_summary']]
        near = [p for p in pool if abs(p['published'] - r['published']) <= window]
        best = {}      # 사건 번호 → 그 사건 기사들과의 가장 높은 유사도
        for p in near:
            best[p['story_id']] = max(best.get(p['story_id'], -1), float(vec @ p['vec']))
        title_key = same_title(r['title'])
        by_title = next((p['story_id'] for p in near if p['title_key'] == title_key), None)
        by_sim = [sid for sim, sid in sorted(((s, sid) for sid, s in best.items() if s >= SIM_HIGH), reverse=True)]
        candidates = list(dict.fromkeys(([by_title] if by_title else []) + by_sim))

        if not candidates:
            story_no += 1
            story_id, joined_by = f'S{story_no:05d}', '새 사건'
        else:
            story_id = min(candidates)          # 가장 먼저 생긴 사건으로 합친다
            joined_by = '유사도' if story_id in by_sim else '제목'
            for other in candidates:
                if other == story_id:
                    continue
                merges.append({'time_utc': now, 'key': r['key'], 'from_story': other, 'into_story': story_id})
                stories.loc[stories['story_id'] == other, 'story_id'] = story_id
                for item in pool + rows:
                    if item['story_id'] == other:
                        item['story_id'] = story_id
                touched.discard(other)
        rows.append({'key': r['key'], 'story_id': story_id, 'group': '사건',
                     'best_sim': round(best[story_id], 4) if story_id in best else '', 'joined_by': joined_by})
        pool.append({'story_id': story_id, 'published': r['published'], 'title_key': title_key, 'vec': vec})
        touched.add(story_id)

    if merges:   # 앞 기사들의 사건 번호가 바뀌었으니 전체를 다시 쓴다
        pd.concat([stories, pd.DataFrame(rows)], ignore_index=True).to_csv(STORIES_PATH, index=False, encoding='utf-8-sig')
        append_csv(merges, STORY_MERGES_PATH)
        redo = merge_followup(merges)
    else:
        append_csv(rows, STORIES_PATH)
        redo = 0
    print(f"  사건 묶음: 기사 {len(rows)}건 (사건 {sum(x['group'] == '사건' for x in rows)}, 분석·추천 글 {sum(x['group'] != '사건' for x in rows)}), "
          f"기존 사건에 붙음 {sum(x['joined_by'] in ('유사도', '제목') for x in rows)}건(제목 규칙 {sum(x['joined_by'] == '제목' for x in rows)}건), "
          f"사건 합침 {len(merges)}번" + (f", 다시 판별할 기사 {redo}건" if redo else ''))
    for m in merges:
        print(f"    합침: {m['from_story']} → {m['into_story']} (기사 {m['key']})")
    sizes = read_csv(STORIES_PATH, STORY_COLUMNS).query("story_id != ''")['story_id'].value_counts()
    for story_id, size in sizes[sizes >= BIG_STORY].items():
        if story_id in touched:
            print(f'    확인 필요: {story_id} 기사 {size}건 — 합치기가 연쇄로 이어졌는지 보세요')
    return touched


## 4. 후속 보도 판별

사건 번호는 있지만 아직 후속 판단이 없는 기사를 **회차 순서대로, 회차 안에서 사건마다** 처리한다. 판단 방법은 `P4_후속보도판별_tmp.ipynb`와 같다(기준은 `prompt/followup_prompt.md`).

- 사건의 첫 기사는 `첫 기사`(소식 `N1`)로 두고 기다린다. 두 번째 기사가 들어올 때 첫 기사 본문도 함께 보내 사실 목록을 만든다.
- 앞 회차 기사와 제목이 같은 기사(재게재)는 LLM 없이 `반복`으로 두고 원래 기사의 소식 번호를 붙인다.
- 그 밖의 기사는 회차·사건마다 LLM을 한 번 부른다. 새 진전이면 새 소식 번호(N2, N3…)가 생기고, 반복·세부 추가면 LLM이 고른 `repeats_of`의 소식 번호를 받는다.
- 상태(사건마다 알려진 사실 목록, 소식 수, 기다리는 첫 기사, 지금까지 나온 제목)는 `followup_state.json`에 저장해 다음 회차로 이어진다.
- LLM에는 **원문 본문**을 보낸다(숫자 변환본이 아님). 본문은 `articles.jsonl`이 아니라 `pending_bodies/<key>.txt`에서 읽는다.
- **본문 지우기(`clean_bodies`, 처리할 때마다)**: ① 후속 판단이 끝났고 그 사건이 첫 기사만 기다리는 상태가 아니면 지운다(이후 판별에는 본문 대신 사실 목록을 쓴다). ② 첫 기사만 있는 사건의 기사는 두 번째 기사가 들어와 첫 기사 본문을 보낼 때까지 남긴다. ③ 그래도 남은 본문은 발행 시각이 가장 최근 수집 시각보다 `COMPARE_DAYS + 1`일 이상 앞서면 지운다(그 뒤로는 같은 사건에 붙을 기사가 없다).
- **사건을 합칠 때**(`merge_followup`, 3절에서 부른다): 뒤 사건의 사실 목록·제목·소식 번호를 앞 사건으로 옮기고, 소식 번호는 앞 사건의 소식 수만큼 밀어 이어 붙인다(`followup.csv`의 `news_id`, `repeats_of`도 바꾼다). 뒤 사건이 아직 첫 기사만 있고 사실 목록이 없으면(LLM 판별 전), 그 기사들의 판단을 지우고 합친 사건에서 다시 판별한다.

In [ ]:
FOLLOWUP_PROMPT = FOLLOWUP_PROMPT_PATH.read_text(encoding='utf-8')
FOLLOWUP_SCHEMA = {
    'type': 'object',
    'properties': {'baseline_facts': {'type': 'array', 'items': {'type': 'string'}},
                   'articles': {'type': 'array', 'items': {
        'type': 'object',
        'properties': {'index': {'type': 'integer'},
                       'decision': {'type': 'string', 'enum': ['새 진전', '세부 추가', '반복', '다른 사건']},
                       'repeats_of': {'type': 'string'},
                       'new_facts': {'type': 'array', 'items': {'type': 'string'}},
                       'reason': {'type': 'string'}},
        'required': ['index', 'decision', 'repeats_of', 'new_facts', 'reason'],
        'additionalProperties': False}}},
    'required': ['baseline_facts', 'articles'],
    'additionalProperties': False,
}
FOLLOWUP_COLUMNS = ['key', 'story_id', 'round', 'decision', 'news_id', 'repeats_of', 'new_facts', 'reason']


def read_body(key):
    path = PENDING_BODY_DIR / f'{key}.txt'
    if not path.exists():
        print('  본문 없음(이미 지웠거나 저장되지 않음):', key)
        return ''
    return path.read_text(encoding='utf-8')


def article_for_llm(row):
    return {'title': row['title'], 'outlet': row['outlet'], 'published_utc': row['published_utc'], 'body': read_body(row['key'])}


def judge(known_facts, baseline_row, new_rows):
    message = {'known_facts': known_facts,
               'baseline_article': None if baseline_row is None else article_for_llm(baseline_row),
               'new_articles': [{'index': i, **article_for_llm(r)} for i, (_, r) in enumerate(new_rows.iterrows(), 1)]}
    text, usage = ask_openai(FOLLOWUP_PROMPT, json.dumps(message, ensure_ascii=False),
                             {'type': 'json_schema', 'name': 'followup', 'strict': True, 'schema': FOLLOWUP_SCHEMA})
    log_usage('followup', new_rows['story_id'].iloc[0], usage)
    return json.loads(text)


def load_followup_state():
    if FOLLOWUP_STATE_PATH.exists():
        return json.loads(FOLLOWUP_STATE_PATH.read_text(encoding='utf-8'))
    return {'known_facts': {}, 'news_count': {}, 'waiting_first': {}, 'seen_titles': {}, 'news_of': {}}


def update_followup():
    articles = load_articles()
    if articles.empty:
        return 0
    stories = read_csv(STORIES_PATH, STORY_COLUMNS)
    done = read_csv(FOLLOWUP_PATH, FOLLOWUP_COLUMNS)
    todo = articles.merge(stories[stories['story_id'] != ''], on='key')
    todo = todo[~todo['key'].isin(done['key'])].sort_values(['round', 'story_id', 'published'])
    if todo.empty:
        return 0
    by_key = articles.set_index('key')
    state = load_followup_state()
    known, count, waiting, titles, news_of = (state[k] for k in ('known_facts', 'news_count', 'waiting_first', 'seen_titles', 'news_of'))
    calls = 0

    for (round_id, story_id), rows in todo.groupby(['round', 'story_id'], sort=True):
        out = []   # 이번 회차·사건의 판단 결과(기사마다 한 줄)
        if story_id not in known:
            # 처음 나온 사건: 첫 기사는 소식 N1. 두 번째 기사가 들어올 때 함께 보낸다.
            first = rows.iloc[0]
            known[story_id], count[story_id], waiting[story_id] = [], 1, first['key']
            titles[story_id] = {same_title(first['title']): first['key']}
            news_of[first['key']] = 'N1'
            out.append({'key': first['key'], 'decision': '첫 기사', 'news_id': 'N1', 'repeats_of': '', 'new_facts': [], 'reason': ''})
            rows = rows.iloc[1:]

        # 앞 회차 기사와 제목이 같으면(다른 매체 재게재) LLM 없이 반복, 원래 기사의 소식 번호
        is_reprint = rows['title'].map(same_title).isin(titles[story_id])
        for _, r in rows[is_reprint].iterrows():
            news_id = news_of.get(titles[story_id][same_title(r['title'])])
            news_of[r['key']] = news_id
            out.append({'key': r['key'], 'decision': '반복', 'news_id': news_id, 'repeats_of': '', 'new_facts': [], 'reason': '앞 기사와 같은 제목(재게재)'})
        rows = rows[~is_reprint]
        for _, r in rows.iterrows():
            titles[story_id].setdefault(same_title(r['title']), r['key'])

        if not rows.empty:
            calls += 1
            baseline_key = waiting.pop(story_id, None)
            answer = judge(known[story_id], None if baseline_key is None else {**by_key.loc[baseline_key].to_dict(), 'key': baseline_key}, rows)
            known[story_id] += [{'news_id': 'N1', 'fact': f} for f in answer['baseline_facts']]
            this_round = {}   # 'A1', 'A2'… → 이번 회차 기사의 소식 번호
            for result, (_, r) in zip(answer['articles'], rows.iterrows()):
                if result['decision'] == '새 진전':
                    count[story_id] += 1
                    news_id = f'N{count[story_id]}'
                elif result['decision'] in ('반복', '세부 추가'):
                    target = result['repeats_of']
                    news_id = this_round.get(target) if target.startswith('A') else (target or None)
                else:   # 다른 사건
                    news_id = None
                this_round[f"A{result['index']}"] = news_id
                news_of[r['key']] = news_id
                known[story_id] += [{'news_id': news_id, 'fact': f} for f in result['new_facts']]
                out.append({'key': r['key'], 'decision': result['decision'], 'news_id': news_id, 'repeats_of': result['repeats_of'],
                            'new_facts': result['new_facts'], 'reason': result['reason']})

        append_csv([{**o, 'story_id': story_id, 'round': round_id, 'new_facts': json.dumps(o['new_facts'], ensure_ascii=False)}
                    for o in out], FOLLOWUP_PATH)
        FOLLOWUP_STATE_PATH.write_text(json.dumps(state, ensure_ascii=False, indent=1), encoding='utf-8')
    print(f'  후속 판별: 기사 {len(todo)}건, LLM 호출 {calls}회')
    return calls


def renumber(news_id, offset):
    # 'N2' → 'N{2 + offset}'. 'A1'(이번 회차 기사)이나 빈 값은 그대로 둔다.
    m = re.fullmatch(r'N(\d+)', news_id or '')
    return f'N{int(m.group(1)) + offset}' if m else news_id


def merge_followup(merges):
    # 3절에서 사건을 합칠 때 부른다. 다시 판별해야 하는 기사 수를 돌려준다.
    state = load_followup_state()
    known, count, waiting, titles, news_of = (state[k] for k in ('known_facts', 'news_count', 'waiting_first', 'seen_titles', 'news_of'))
    done = read_csv(FOLLOWUP_PATH, FOLLOWUP_COLUMNS)
    redo = 0
    for m in merges:
        old, new = m['from_story'], m['into_story']
        sel = done['story_id'] == old
        if old not in known:                  # 아직 후속 판별 전인 사건: 옮길 것이 없다
            continue
        if new not in known:                  # 합치는 쪽이 판별 전: 상태를 그대로 옮긴다
            for d in (known, count, titles):
                d[new] = d.pop(old)
            if old in waiting:
                waiting[new] = waiting.pop(old)
            done.loc[sel, 'story_id'] = new
            continue
        if old in waiting:                    # 첫 기사만 있고 사실 목록이 없는 사건: 합친 사건에서 다시 판별
            waiting.pop(old)
            for key in done.loc[sel, 'key']:
                news_of.pop(key, None)
            redo += int(sel.sum())
            done = done[~sel]
            for d in (known, count, titles):
                d.pop(old)
            continue
        offset = count[new]
        known[new] += [{'news_id': renumber(f['news_id'], offset), 'fact': f['fact']} for f in known.pop(old)]
        count[new] += count.pop(old)
        for title, key in titles.pop(old).items():
            titles[new].setdefault(title, key)
        for key in done.loc[sel, 'key']:
            news_of[key] = renumber(news_of.get(key), offset)
        done.loc[sel, 'news_id'] = done.loc[sel, 'news_id'].map(lambda x: renumber(x, offset))
        done.loc[sel, 'repeats_of'] = done.loc[sel, 'repeats_of'].map(lambda x: renumber(x, offset))
        done.loc[sel, 'story_id'] = new
    if FOLLOWUP_PATH.exists():
        done[FOLLOWUP_COLUMNS].to_csv(FOLLOWUP_PATH, index=False, encoding='utf-8-sig')
    if FOLLOWUP_STATE_PATH.exists():
        FOLLOWUP_STATE_PATH.write_text(json.dumps(state, ensure_ascii=False, indent=1), encoding='utf-8')
    return redo


def clean_bodies():
    """후속 판별에 다 쓴 본문과 오래된 본문을 지운다. 지운 수를 돌려준다."""
    files = {p.stem: p for p in PENDING_BODY_DIR.glob('*.txt')} if PENDING_BODY_DIR.exists() else {}
    if not files:
        return 0
    waiting_stories = set(load_followup_state()['waiting_first'])
    done = read_csv(FOLLOWUP_PATH, FOLLOWUP_COLUMNS).drop_duplicates('key', keep='last').set_index('key')['story_id']   # 같은 기사가 두 번 판별된 줄이 있으면 마지막만(2026-11-23)
    articles = load_articles()
    if articles.empty:
        return 0
    latest = pd.to_datetime(articles['first_seen_utc'], utc=True, format='ISO8601').max()
    old = set(articles.loc[articles['published'] < latest - pd.Timedelta(days=COMPARE_DAYS + 1), 'key'])
    gone = [k for k in files if (k in done.index and done[k] not in waiting_stories) or k in old]
    for k in gone:
        files[k].unlink()
    return len(gone)

## 5. 매체 수 갱신

저장된 모든 사건을 다시 센다(계산만 하므로 빠르다). **"서로 다른 원 출처가 몇 곳에서 전했나"**를 센다.

- 기사마다 **센 출처**를 정한다.
  - 야후가 아닌 기사: `outlet`(이 매체들은 페이지에 제공 조직(provider)이 거의 없고, 게재한 곳이 곧 원 출처다)
  - 야후 기사: ① 같은 사건에 제목이 같은 다른 매체 기사가 있으면 그 매체 ② 아니면 `provider`(원 출처, 2026-11-19 저장 원문 786쪽 모두 값이 있었다) ③ provider가 야후 계열(Yahoo Finance, Yahoo Finance Video…)이거나 비어 있으면 야후
  - provider 이름은 대소문자·앞의 "The"·".com"·문장부호를 무시해 같은 곳으로 본다("Motley Fool" = "The Motley Fool"). 우리 수집 매체 이름(`SITE_NAMES`, news_collect와 같은 표)이면 그 도메인으로 바꿔 원본과 같은 곳으로 센다("Motley Fool" → `fool.com`).
- **재게재**: 같은 사건 안에서 센 출처와 제목이 같은 기사는 처음 것만 기사 수에 넣고 나머지는 재게재로 따로 센다(git_code `news_process.ipynb`와 같은 생각).
- 사건 매체 수(`story_outlets`): 사건의 모든 기사(후속 포함)의 서로 다른 센 출처 수. 소식 매체 수(`news_outlets`): 같은 소식 번호 기사들의 서로 다른 센 출처 수.
- 기사 수(`story_articles`, `news_articles`)는 재게재를 뺀 수, 재게재 수는 `story_reposts`, `news_reposts`.
- `update_llm_source()`: 매체 수를 센 뒤 `llm_source.jsonl`의 기사마다 `story_id`(사건 번호)와 `news_outlets`(그 기사가 속한 소식의 매체 수)를 채워 파일을 다시 쓴다. 사건 기사가 아니면 story_id는 빈 값, 사건 기사가 아니거나 후속 판별에서 '다른 사건'이면 news_outlets는 None. 후속 판별 전 기사는 소식 번호 '미정' 묶음의 매체 수.
- 순위: 사건 매체 수 → 사건 기사 수 → 사건의 마지막 발행 시각(최근 먼저) → 소식 번호.
- `다른 사건`으로 판단된 기사는 세지 않는다. 아직 후속 판단 전인 기사는 소식 번호 `미정`으로 센다.
- **기사마다 센 값을 `llm_source.jsonl`에도 남긴다**(2026-11-20, `update_llm_source`): `news_id`(소식 번호, 판별 전이면 `미정`), `followup_decision`(후속 판단), `source_key`·`source_name`(센 출처), `repost`(재게재). 요약 노트북은 이 값으로 **사용자가 정한 시간대 안의 기사만** 다시 세어 순위를 매긴다(위의 매체 수는 전체 기간 값으로 그대로 남긴다). 사건 기사가 아니거나 '다른 사건'이면 빈 값.

In [ ]:
# news_collect.ipynb의 매체 이름 표(우리 수집 매체 11곳). provider 이름을 우리 도메인으로 바꿀 때 쓴다.
SITE_NAMES = {'finance.yahoo.com': 'Yahoo Finance', 'fool.com': 'Motley Fool', 'cnbc.com': 'CNBC', 'theguardian.com': 'Guardian',
              'businessinsider.com': 'Business Insider', 'foxbusiness.com': 'Fox Business', 'fortune.com': 'Fortune',
              'cbsnews.com': 'CBS News', 'nypost.com': 'NY Post', 'usatoday.com': 'USA Today', 'cnn.com': 'CNN'}
EXTRA_SITE_NAMES = {'New York Post': 'nypost.com'}   # SITE_NAMES와 표기가 다른 이름


def name_key(name):
    # 'The Motley Fool' → 'motleyfool', 'GuruFocus.com' → 'gurufocus', 'Globe Newswire' → 'globenewswire'
    name = re.sub(r'^the\s+', '', str(name or '').strip().lower())
    return re.sub(r'[^a-z0-9]', '', re.sub(r'\.com$', '', name))


PROVIDER_TO_SITE = {name_key(n): d for d, n in SITE_NAMES.items()} | {name_key(n): d for n, d in EXTRA_SITE_NAMES.items()}


def source_of(outlet, provider, original_outlet):
    """기사의 센 출처 → (같은 곳인지 비교하는 키, 보여 줄 이름)."""
    if outlet != YAHOO:
        return outlet, outlet
    if original_outlet:                       # 같은 사건에 제목이 같은 다른 매체 기사
        return original_outlet, original_outlet
    key = name_key(provider)
    if not key or key.startswith('yahoo'):    # 야후 자체 기사 또는 원 출처를 모름
        return YAHOO, YAHOO
    site = PROVIDER_TO_SITE.get(key)
    return (site, site) if site else (key, str(provider).strip())


def name_list(names):
    return ', '.join(sorted(set(names)))


def count_sources():
    """사건 기사마다 센 출처·재게재·소식 번호('다른 사건'은 뺀다). update_importance와 update_llm_source가 함께 쓴다."""
    articles = load_articles()
    stories = read_csv(STORIES_PATH, STORY_COLUMNS)
    followup = read_csv(FOLLOWUP_PATH, FOLLOWUP_COLUMNS)[['key', 'decision', 'news_id']]
    data = articles.merge(stories[stories['story_id'] != ''], on='key').merge(followup, on='key', how='left')
    if data.empty:
        return pd.DataFrame()
    data = data[data['decision'] != '다른 사건'].copy()
    data['news_id'] = data['news_id'].replace('', np.nan).fillna('미정')
    data['provider'] = data['provider'].fillna('') if 'provider' in data else ''

    # 센 출처
    data['title_key'] = data['title'].map(same_title)
    not_yahoo = data[data['outlet'] != YAHOO]
    original = {(s, t): o for s, t, o in zip(not_yahoo['story_id'], not_yahoo['title_key'], not_yahoo['outlet'])}
    counted = [source_of(o, p, original.get((s, t)))
               for o, p, s, t in zip(data['outlet'], data['provider'], data['story_id'], data['title_key'])]
    data['source_key'] = [k for k, _ in counted]
    data['source_name'] = [n for _, n in counted]
    # 재게재: 같은 사건에서 센 출처·제목이 같은 기사 중 처음 것이 아닌 것
    data = data.sort_values('published')
    data['repost'] = data.duplicated(['story_id', 'source_key', 'title_key'])
    data['is_start'] = data['decision'].isin(['첫 기사', '새 진전'])
    return data


def update_importance(data=None):
    data = count_sources() if data is None else data
    if data.empty:
        return pd.DataFrame()

    story = data.groupby('story_id').agg(story_outlets=('source_key', 'nunique'),
                                         story_articles=('repost', lambda x: int((~x).sum())), story_reposts=('repost', 'sum'),
                                         story_outlet_list=('source_name', name_list),
                                         story_last_published=('published_utc', 'max')).reset_index()
    news = (data.sort_values(['is_start', 'published_utc'], ascending=[False, True])
            .groupby(['story_id', 'news_id']).agg(title=('title', 'first'), short_summary=('short_summary', 'first'),
                                                  first_published=('published_utc', 'min'),
                                                  news_outlets=('source_key', 'nunique'),
                                                  news_articles=('repost', lambda x: int((~x).sum())), news_reposts=('repost', 'sum'),
                                                  news_outlet_list=('source_name', name_list)).reset_index())
    importance = news.merge(story, on='story_id')
    importance['news_no'] = importance['news_id'].str.extract(r'(\d+)', expand=False).astype(float)
    importance = (importance.sort_values(['story_outlets', 'story_articles', 'story_last_published', 'story_id', 'news_no'],
                                         ascending=[False, False, False, True, True])
                  .drop(columns='news_no').reset_index(drop=True))
    importance.to_csv(IMPORTANCE_PATH, index=False, encoding='utf-8-sig')
    print(f"  매체 수: 사건 {importance['story_id'].nunique()}개, 소식 {len(importance)}개 →", IMPORTANCE_PATH.name)
    return importance


def update_llm_source(importance=None, data=None):
    """llm_source.jsonl(news_extract가 쌓음)의 기사마다 story_id(사건 번호)와 news_outlets(그 기사가 속한 소식의 매체 수)를 채워 다시 쓴다.
    사건에 묶이지 않은 기사(사건 기사가 아님)는 story_id = '', news_outlets = None. 후속 판별에서 '다른 사건'이면 news_outlets = None.
    기사마다 센 값(news_id, followup_decision, source_key, source_name, repost)도 채운다(요약 노트북이 시간대 안에서 다시 센다)."""
    if not LLM_SOURCE_PATH.exists():
        return 0
    raw = LLM_SOURCE_PATH.read_bytes()
    cut = raw.rfind(b'\n') + 1                     # 다 쓰인 줄까지만 고친다(쓰는 중인 마지막 줄은 그대로 뒤에 붙인다)
    rows = [json.loads(l) for l in raw[:cut].decode('utf-8').splitlines() if l.strip()]
    if importance is None:
        importance = read_csv(IMPORTANCE_PATH, [])
    story = dict(zip(*read_csv(STORIES_PATH, STORY_COLUMNS)[['key', 'story_id']].values.T)) if STORIES_PATH.exists() else {}
    fu = read_csv(FOLLOWUP_PATH, FOLLOWUP_COLUMNS)
    followup = {k: (d, n) for k, d, n in zip(fu['key'], fu['decision'], fu['news_id'])}
    data = count_sources() if data is None else data
    counted = {} if data.empty else {k: (n, s, sn, bool(rp)) for k, n, s, sn, rp in
                                      zip(data['key'], data['news_id'], data['source_key'], data['source_name'], data['repost'])}
    outlets = {} if importance.empty else {(s, n): int(o) for s, n, o in
                                           zip(importance['story_id'], importance['news_id'], importance['news_outlets'])}
    filled = 0
    for r in rows:
        sid = story.get(r['key'], '')
        decision, news_id = followup.get(r['key'], ('', ''))
        r['story_id'] = sid
        r['news_outlets'] = None if not sid or decision == '다른 사건' else outlets.get((sid, news_id or '미정'))
        n, s, sn, rp = counted.get(r['key'], (None, None, None, None))
        r.update(news_id=n, followup_decision=decision or None, source_key=s, source_name=sn, repost=rp)
        filled += r['news_outlets'] is not None
    text = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows).encode('utf-8')
    tmp = LLM_SOURCE_PATH.with_suffix('.jsonl.tmp')
    tmp.write_bytes(text + LLM_SOURCE_PATH.read_bytes()[cut:])   # 읽은 뒤 news_extract가 붙인 줄도 잃지 않는다
    tmp.replace(LLM_SOURCE_PATH)
    print(f'  llm_source: {len(rows)}건 중 사건 기사 {sum(bool(r["story_id"]) for r in rows)}건, 소식 매체 수 {filled}건 채움')
    return filled

## 6. 한 번 처리하기

아직 사건 번호가 없는 추출 기사가 있으면 사건 묶음 → 후속 보도 판별 → 매체 수를 차례로 하고, `track_log.csv`에 한 줄 남긴다.

In [ ]:
def pending_count():
    articles = pd.DataFrame(read_jsonl(ARTICLES_PATH))
    if articles.empty:
        return 0
    articles = articles.drop_duplicates('key', keep='last')   # 같은 기사는 마지막 줄만(load_articles와 같게)
    stories = read_csv(STORIES_PATH, STORY_COLUMNS)
    return int((articles['extraction'].notna() & ~articles['key'].isin(stories['key'])).sum())


def run_once():
    pending = pending_count()
    if not RUN_LLM:
        print(f'사건 묶음을 기다리는 추출 기사 {pending}건 — RUN_LLM = False라 처리하지 않는다')
        return
    start = time.time()
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')
    touched = update_stories()
    calls = update_followup()
    deleted = clean_bodies()
    if deleted:
        print(f'  본문 지움: {deleted}건 (남은 임시 본문 {len(list(PENDING_BODY_DIR.glob("*.txt")))}건)')
    if touched or calls:
        data = count_sources()
        update_llm_source(update_importance(data), data)
        append_csv([{'time_utc': now, 'articles': pending, 'stories_touched': len(touched), 'followup_calls': calls,
                     'seconds': round(time.time() - start, 1)}], TRACK_LOG_PATH)


run_once()

## 7. 대기하며 돌리기

`WATCH_RUN = True`로 켜 두면 `WATCH_SECONDS`초마다 새 추출 결과를 확인해 처리한다. `news_extract_tmp.ipynb`와 함께 켜 둔다. 오류가 나도 멈추지 않고 다음에 다시 한다. 멈추려면 커널을 중단한다(■).

In [ ]:
if WATCH_RUN:
    print(f'{WATCH_SECONDS}초마다 새 추출 결과를 확인합니다. 멈추려면 커널을 중단하세요.')
    while True:
        try:
            run_once()
        except Exception as e:
            print('오류:', type(e).__name__, str(e)[:200], '(다음에 다시 시도)')
        time.sleep(WATCH_SECONDS)
else:
    print('WATCH_RUN = False: 대기하며 돌리지 않는다.')

## 8. 결과 보기

`show_story('S00001')`: 사건 하나를 수집 회차 순서로, 판단·소식 번호·새 사실과 함께 보여 준다.

In [ ]:
def show_story(story_id):
    data = (load_articles().merge(read_csv(STORIES_PATH, STORY_COLUMNS), on='key')
            .merge(read_csv(FOLLOWUP_PATH, FOLLOWUP_COLUMNS).drop(columns=['story_id', 'round']), on='key', how='left'))
    for _, r in data[data['story_id'] == story_id].sort_values(['round', 'published']).iterrows():
        print(f"[{r['decision'] or '판단 전'} {r['news_id'] or ''}] 회차 {r['round']} / 발행 {r['published_utc'][:16]} {r['outlet']} | {r['title']}")
        for fact in json.loads(r['new_facts']) if isinstance(r['new_facts'], str) and r['new_facts'] else []:
            print('    새 사실:', fact)


if IMPORTANCE_PATH.exists():
    display(pd.read_csv(IMPORTANCE_PATH).head(20))